In [ ]:
# James Caldwell, UVA IRA
# 12/17/24, updated 5/15/26
# Report for generating demographics for the Public Service Pathways program (several hundred students)
   # Gender, race, year, first generation status, residency, school
# Data came from qlik with some sorting done in excel before importing, cleaning, and calculating percentage breakdowns.

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

enrollment_file_path = os.getenv('fall_2025_enrollment') 
pathway_list = os.getenv('pathway_list')
psp_students = os.getenv('psp_students')

# data = pd.read_excel(enrollment_file_path) # Need to update this filepath in the .env file if using again
pathway_list = pd.read_csv(pathway_list)
psp_students = pd.read_excel(psp_students)
psp_students_no_duplicates = psp_students.copy(deep=True)

In [10]:
df_cleaned = data.drop_duplicates(subset='Email Address UVA',keep='last')
data_filtered = df_cleaned.iloc[:, 1:]

In [11]:
# Function to calculate unique entries and percentages
def get_unique_values_with_percentages(df):
    result = {}
    for column in df.columns:
        value_counts = df[column].value_counts(normalize=True)
        result[column] = value_counts.reset_index().rename(columns={'index': 'Value', column: 'Percentage'})
    return result

# Get percentages
unique_values_percentages = get_unique_values_with_percentages(data_filtered)


In [13]:
# Display group breakdowns
i = 1
for column, data in unique_values_percentages.items():
    if i == 8: # Adjust 4 through 7 here to print out different groups
        print(f"Column: {column}")
        print(data)
        data.to_clipboard(index=False) # Data is copied to clipboard, ready to paste into Excel
        print()
    i = i + 1

Column: Pell Recipient Flag
                  Value  Percentage
0  Not a Pell Recipient    0.801441
1        Pell Recipient    0.198559



In [14]:
i = 1
for column, data in unique_values_percentages.items():
    if i == 4:  # adjust as needed
        print(f"Column: {column}")
        print(data.to_csv(sep='\t', index=False))  # tab-separated for Excel
        data.to_clipboard(index=False)

    i += 1

Column: Primary Acad Program School
Value	Percentage
Arts and Sciences	54.976303317535546
Engineering	25.497630331753555
Architecture	5.876777251184834
Education	4.170616113744076
Nursing	2.938388625592417
Leadership and Public Policy	2.938388625592417
Commerce	2.3696682464454977
Data Science	0.6635071090047393
Medicine Graduate	0.37914691943127965
Medicine	0.18957345971563982



Spring 2026 add: calculating major breakdowns as a whole and by pathway

In [3]:
pathway_list.rename(columns={'email': 'Email Address UVA'}, inplace=True)

In [4]:
pathway_list['Email Address UVA'].duplicated().sum()

122

In [5]:
psp_students = pd.merge(psp_students, pathway_list, on='Email Address UVA', how='left')
    # This will have some duplicate rows on SSID since some students have multiple pathways

In [12]:
counts = psp_students['Pathway'].value_counts(dropna=False)
percents = psp_students['Pathway'].value_counts(dropna=False, normalize=True) * 100

summary = pd.DataFrame({
    'Count': counts,
    'Percent': percents.round(2)
})

summary.to_csv('psp_students_pathway_summary.csv', index=True)

In [12]:
psp_students.columns

Index(['Primary Name', 'Email Address UVA', 'Student System ID',
       'Email Address UVA.1', 'Primary Acad Program School', 'Gender',
       'IPEDS Race', 'Residency', 'Pell Recipient Flag', 'First Generation',
       'Admit Type Desc', 'Primary Major (Collapsed)', 'ComputingID',
       'Pathway'],
      dtype='object')

In [11]:
counts = psp_students_no_duplicates['Primary Major (Collapsed)'].value_counts(dropna=False)
percents = psp_students_no_duplicates['Primary Major (Collapsed)'].value_counts(dropna=False, normalize=True) * 100

summary = pd.DataFrame({
    'Count': counts,
    'Percent': percents.round(2)
})

summary.to_csv('psp_students_major_summary.csv', index=True)

In [13]:
summary = psp_students.pivot_table(
    index="Primary Major (Collapsed)",
    columns="Pathway",
    aggfunc='size',
    fill_value=0
)
summary.to_csv('psp_students_major_pathway_summary.csv', index=True)